# 07 — Model Comparison
Runs `compare_models.py`'s GroupKFold cross-validation (grouped by `unit_id`) across all registered models and visualizes the result. This is the notebook to actually make a model choice from — `05_model_baselines.ipynb` uses a single split and can be noisy.

In [ ]:
import sys, os
sys.path.append(os.path.abspath('..'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

import config
import preprocessing
import feature_engineering as fe
import compare_models

plt.rcParams['figure.figsize'] = (10, 4)

## Load features

In [ ]:
if not os.path.exists(config.FEATURES_DATA_PATH):
    preprocessing.run_preprocessing()
    feat_df = fe.run_feature_engineering()
else:
    feat_df = pd.read_csv(config.FEATURES_DATA_PATH, parse_dates=[config.COL_TIMESTAMP])

feature_cols = fe.get_feature_columns(feat_df)
X = feat_df[feature_cols]
y = feat_df[config.COL_RUL]
groups = feat_df[config.COL_UNIT_ID]

print(f'{groups.nunique()} units, {len(feat_df)} rows, {len(feature_cols)} features')

## Run GroupKFold comparison across all models
Reuses `compare_models.run_cv_for_model` directly — same folds, same metrics as running `python compare_models.py` from the command line, just kept in memory here for plotting.

In [ ]:
from models import MODEL_REGISTRY

fold_results = {}
for name, model_cls in MODEL_REGISTRY.items():
    print(f'Running {name}...')
    fold_results[name] = compare_models.run_cv_for_model(model_cls, X, y, groups, feature_cols)

## Summary table (mean +/- std across folds)

In [ ]:
summary_rows = []
for name, fold_df in fold_results.items():
    means = fold_df.drop(columns='fold').mean()
    stds = fold_df.drop(columns='fold').std()
    row = {f'{k}_mean': means[k] for k in means.index}
    row.update({f'{k}_std': stds[k] for k in stds.index})
    row['model'] = name
    summary_rows.append(row)

summary_df = pd.DataFrame(summary_rows).set_index('model')
summary_df = summary_df.sort_values('PHM08_mean')
summary_df

## Compare models by metric
Bar chart with error bars (std across folds) for each metric. Watch for models whose error bars overlap heavily — a difference in mean score that's smaller than the fold-to-fold variance isn't a real difference, especially with few units.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(11, 9))
for ax, metric in zip(axes.ravel(), ['MAE', 'RMSE', 'R2', 'PHM08']):
    means = summary_df[f'{metric}_mean']
    stds = summary_df[f'{metric}_std']
    ax.bar(means.index, means.values, yerr=stds.values, capsize=4)
    ax.set_title(metric)
    ax.tick_params(axis='x', rotation=30)
plt.tight_layout()
plt.show()

## Fold-by-fold detail
Per-fold PHM08 scores — useful for spotting whether one particular unit (fold) is dragging a model's average down, which single-number summaries hide.

In [ ]:
fold_phm08 = pd.DataFrame({name: df.set_index('fold')['PHM08'] for name, df in fold_results.items()})
fold_phm08

In [ ]:
fold_phm08.plot(kind='bar', figsize=(10, 4))
plt.title('PHM08 score by fold and model (lower is better)')
plt.ylabel('PHM08 score')
plt.xlabel('fold')
plt.tight_layout()
plt.show()

## Decision
Pick the model with the best PHM08 mean *and* acceptable variance across folds — not just the lowest mean. If the best model's std spans into the second-best model's mean, treat them as effectively tied and prefer the simpler one. Once decided, run:

```bash
python train.py --model <chosen_model>
```

to produce the final saved artifact for `predict.py`. Only build out `models/stacking.py` from here if the gap between the top single model and a hypothetical ensemble seems worth the added complexity — see the caution in that file's docstring.